# 00 · Smoke Test — Data Splits

Quick sanity check that the **loan-stratified temporal split** is usable end-to-end, *before*
investing in the foundation model. Trains an XGBoost baseline to confirm:

1. `data/processed/{train,val,test}.parquet` load and align with `tokenizer.yaml`.
2. A forward-looking default label can be built.
3. There is real predictive signal (AUC >> 0.5), and **val ~= test** (the temporal split generalizes).

**Prerequisite:** run the split first — `python scripts/prepare_data.py --input data/raw/all_cutoffs.parquet`.

In [ ]:
import yaml
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

cfg = yaml.safe_load(open('../configs/dutch_mortgages/tokenizer.yaml'))
NUM = cfg['profile'].get('numeric', []) + cfg['event'].get('numeric', [])
CAT = (cfg['profile'].get('categorical', []) + cfg['profile'].get('flags', [])
       + cfg['event'].get('categorical', []) + cfg['event'].get('flags', []))
FEATS = NUM + CAT
print(f'{len(FEATS)} features ({len(NUM)} numeric, {len(CAT)} categorical)')

## Label: CRR default within the next 6 months

Observe each loan at **month 12** (`2024-12-31`); label = 1 if `default_crr_flag` becomes `Y`
in any of the next 6 monthly cutoffs. One row per loan -> matches the loan-level split.

In [ ]:
OBS = '2024-12-31'
FWD = ['2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30', '2025-05-31', '2025-06-30']
COLS = list(set(FEATS + ['loan_id', 'reporting_date', 'default_crr_flag']))

def build(path):
    df = pd.read_parquet(path, columns=COLS)
    obs = df[df.reporting_date == OBS].copy()
    fut = df[df.reporting_date.isin(FWD)]
    defaulted = set(fut.loc[fut.default_crr_flag == 'Y', 'loan_id'])
    y = obs.loan_id.isin(defaulted).astype(int).values
    return obs[FEATS].copy(), y

Xtr, ytr = build('../data/processed/train.parquet')
Xva, yva = build('../data/processed/val.parquet')
Xte, yte = build('../data/processed/test.parquet')
print(f'rows  train {len(ytr):,}  val {len(yva):,}  test {len(yte):,}')
print(f'default rate  train {ytr.mean():.3%}  val {yva.mean():.3%}  test {yte.mean():.3%}')

In [ ]:
cats_map: dict = {}
def encode(X, fit=False):
    X = X.copy()
    for c in CAT:
        if fit:
            cats_map[c] = {v: i for i, v in enumerate(pd.Series(X[c].dropna().unique()))}
        X[c] = X[c].map(cats_map[c]).astype('float')
    for c in NUM:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    return X

Xtr, Xva, Xte = encode(Xtr, fit=True), encode(Xva), encode(Xte)
model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
                          n_jobs=-1, tree_method='hist')
model.fit(Xtr, ytr)

In [ ]:
for name, X, y in [('val', Xva, yva), ('test', Xte, yte)]:
    p = model.predict_proba(X)[:, 1]
    print(f'{name:>4}:  ROC-AUC {roc_auc_score(y, p):.4f}   PR-AUC {average_precision_score(y, p):.4f}')

## What this is (and isn't)

This is a **smoke test** — config (1) of the baseline: *all features, no gate*. On
`all_cutoffs.parquet` it gives **ROC-AUC ~= 0.93, PR-AUC ~= 0.81** (default rate ~4%), val ~= test.
That high number is **inflated by leakage**: contemporaneous `arrears_bucket`/`performing_status`/
`default_crr_flag` basically read the current delinquency state.

**The honest baseline is NOT this number.** See `scripts/train_baseline.py` /
`reports/baseline_report.md`, which run four configurations and isolate the real
**Gate G1 = ROC-AUC ~= 0.73, PR-AUC ~= 0.046** (config 4: no-leakage features + performing-at-obs
gate — predict *new* defaults among currently-performing loans). That hard, low-base-rate task is
the room a foundation model can exploit.

Use this notebook only to confirm the splits load, align with `tokenizer.yaml`, and carry signal —
not as the baseline.